In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls "/content/drive/MyDrive/EMBS model train"

archive.zip


In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/EMBS model train/archive.zip"
extract_path = "/content/ravdess"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    for file in zip_ref.namelist():
        try:
            zip_ref.extract(file, extract_path)
        except:
            pass

print("Extraction Completed!")

Extraction Completed!


In [ ]:
!find /content/ravdess -name "*.wav" | wc -l

2880


In [ ]:
import os
import librosa
import numpy as np

In [ ]:
!pip install librosa -q

In [ ]:
emotion_map = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised"
}

In [ ]:
def extract_features(file_path):
    audio, sr = librosa.load(file_path, duration=3, offset=0.5)

    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=sr,
        n_mfcc=40
    )

    return np.mean(mfcc.T, axis=0)

In [ ]:
X = []
y = []

for root, dirs, files in os.walk("/content/ravdess"):
    for file in files:
        if file.endswith(".wav"):

            emotion_code = file.split("-")[2]

            if emotion_code in emotion_map:

                path = os.path.join(root, file)

                try:
                    features = extract_features(path)

                    X.append(features)
                    y.append(emotion_map[emotion_code])

                except:
                    pass

print("Total Samples:", len(X))

Total Samples: 2880


In [ ]:
X = np.array(X)

print("Feature Shape:", X.shape)
print("Total Labels:", len(y))

Feature Shape: (2880, 40)
Total Labels: 2880


In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y = encoder.fit_transform(y)

print("Classes:", encoder.classes_)

Classes: ['angry' 'calm' 'disgust' 'fearful' 'happy' 'neutral' 'sad' 'surprised']


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Set:", X_train.shape)
print("Testing Set:", X_test.shape)

Training Set: (2304, 40)
Testing Set: (576, 40)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

print("Training Complete!")

Training Complete!


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))

print("\nClassification Report:\n")
print(classification_report(
    y_test,
    pred,
    target_names=encoder.classes_
))

Accuracy: 0.9305555555555556

Classification Report:

              precision    recall  f1-score   support

       angry       0.97      1.00      0.99        76
        calm       0.97      1.00      0.99        77
     disgust       0.89      0.87      0.88        77
     fearful       0.83      0.90      0.86        77
       happy       1.00      0.87      0.93        77
     neutral       0.95      1.00      0.97        38
         sad       0.92      0.92      0.92        77
   surprised       0.92      0.92      0.92        77

    accuracy                           0.93       576
   macro avg       0.93      0.94      0.93       576
weighted avg       0.93      0.93      0.93       576



In [ ]:
import joblib

joblib.dump(model, "ravdess_emotion_model.pkl")

print("Model Saved Successfully!")

Model Saved Successfully!


In [ ]:
from google.colab import files

files.download("ravdess_emotion_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving 03-01-08-01-01-01-22.wav to 03-01-08-01-01-01-22.wav


In [ ]:
audio_path = list(uploaded.keys())[0]

print(audio_path)

03-01-08-01-01-01-22.wav


In [ ]:
features = extract_features(audio_path)

prediction = model.predict([features])

emotion = encoder.inverse_transform(prediction)

print("Predicted Emotion:", emotion[0])

# Confidence Percentage
probabilities = model.predict_proba([features])

confidence = np.max(probabilities) * 100

print("Confidence: {:.2f}%".format(confidence))



Predicted Emotion: surprised
Confidence: 70.00%


In [ ]:
from google.colab import files

files.download("ravdess_emotion_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>